# Домашнее задание № 2
Односвязные списки, строки и метод двух указателей. Python.

Использую `Node` из лекции: узел хранит значение и ссылку `next`. Функции `make_list` и `to_list` нужны только для примеров, в самих алгоритмах новые списки не создаю.

In [1]:
class Node:
    # Узел — отдельный элемент связного списка, а не весь список целиком.
    def __init__(self, val, next=None):
        self.val = val  # Значение, которое хранится в этом узле.
        self.next = next  # Ссылка на следующий узел; None означает конец списка.


def make_list(values):
    head = None
    # Каждый новый узел ставим перед текущей головой.
    # Поэтому идём с конца: для [1, 2, 3] получим 3, затем 2 -> 3, затем 1 -> 2 -> 3.
    for x in reversed(values):
        head = Node(x, head)
    return head  # Достаточно вернуть голову: остальные узлы доступны через next.


def to_list(head):
    # Вспомогательная функция для вывода: собираем значения узлов в обычный список.
    result = []
    while head:
        result.append(head.val)
        head = head.next  # Переходим к следующему узлу, пока не дойдём до None.
    return result

## 1. Развернуть односвязный список
В `nxt` сохраняю следующий узел, затем ставлю `cur.next = prev`. Сдвигаю `prev` и `cur`. Так разворачиваются все ссылки. **Время O(n), память O(1)**. Семинар, стр. 8–12.

In [2]:
def reverse_list(head):
    # prev — голова уже развёрнутой части, cur — первый ещё не обработанный узел.
    # Сначала развёрнутая часть пуста, а cur указывает на начало исходного списка.
    prev = None
    cur = head
    while cur:
        # Сохраняем следующий узел ДО изменения ссылки, чтобы не потерять остаток списка.
        nxt = cur.next
        cur.next = prev  # Текущий узел теперь смотрит назад, в развёрнутую часть.
        prev = cur  # Этот узел стал новой головой развёрнутой части.
        cur = nxt  # Продолжаем обработку сохранённого остатка.
    # Например, 1 -> 2 -> 3 превращается в 3 -> 2 -> 1; бывшая голова смотрит на None.
    # Когда cur равен None, все узлы обработаны и новая голова находится в prev.
    return prev

print(to_list(reverse_list(make_list([1, 2, 3, 4]))))

[4, 3, 2, 1]


## 2. Найти середину списка
`slow` идёт на один узел, а `fast` на два. Когда `fast` дошёл до конца, `slow` находится в середине. При чётной длине получается вторая середина. **Время O(n), память O(1)**. Семинар, стр. 13–18.

In [3]:
def find_middle(head):
    # Оба указателя начинают с головы, но движутся с разной скоростью.
    slow = fast = head
    # Нужны и fast, и fast.next, чтобы безопасно сделать два перехода.
    while fast and fast.next:
        slow = slow.next  # Один шаг за итерацию.
        fast = fast.next.next  # Два шага за ту же итерацию.
    # Пока fast проходит весь список, slow успевает пройти только половину.
    # Для 1 -> 2 -> 3 -> 4 вернётся 3: при чётной длине выбираем вторую середину.
    # У пустого списка slow сразу равен None, цикл не выполняется.
    return slow

print(find_middle(make_list([1, 2, 3, 4, 5])).val)

3


## 3. Удалить элемент
Удаляю **все** узлы со значением `val`, как в семинаре. Сначала передвигаю `head` через совпадения в начале. Далее пропускаю совпадения через `cur.next`. Новых узлов нет. **Время O(n), память O(1)**. Семинар, стр. 19–23.

In [4]:
def remove_value(head, val):
    # У головы нет предыдущего узла, через который можно было бы изменить ссылку.
    # Поэтому сначала сдвигаем саму голову, пока в начале стоят удаляемые значения.
    while head and head.val == val:
        head = head.next

    cur = head
    # Проверяем узел ПОСЛЕ cur: так у нас есть ссылка, которую нужно перенаправить.
    while cur and cur.next:
        if cur.next.val == val:
            # Было cur -> удаляемый -> следующий; станет cur -> следующий.
            # cur оставляем на месте: новый сосед тоже может содержать val.
            cur.next = cur.next.next
        else:
            # Соседа оставляем в списке и переходим к нему для проверки дальше.
            cur = cur.next
    # Голова могла измениться; если удалили все узлы, здесь будет None.
    return head

print(to_list(remove_value(make_list([1, 2, 3, 2, 4]), 2)))

[1, 3, 4]


## 4. Является ли `a` исходной для `b`
Двигаю указатель по `b`; указатель по `a` сдвигаю при совпадении. Если дошёл до конца `a`, порядок букв сохранился. **Время O(len(b)), память O(1)**. Семинар, стр. 41–47.

In [5]:
def is_subsequence(a, b):
    # Проверяем, можно ли получить a из b удалением символов без перестановки.
    # i — индекс следующего нужного символа a и число уже найденных символов.
    i = 0
    for ch in b:
        # Совпадение продвигает нас по a, неподходящий символ b просто пропускаем.
        # Проверка i < len(a) защищает от выхода за границы, если a уже найдена целиком.
        if i < len(a) and a[i] == ch:
            i += 1
    # Для a='abc', b='axbcy' символы x и y пропускаем, а i вырастает до 3.
    # Успех только тогда, когда найдены все символы a. Пустая a подходит всегда.
    return i == len(a)

print(is_subsequence('abc', 'axbcy'))

True


## 5. Поиск пары в отсортированном массиве
Беру левый и правый элементы. Если сумма меньше `target`, сдвигаю левый указатель, иначе — правый. Из-за сортировки подходящая пара не теряется. **Время O(n), память O(1)**.

In [6]:
def two_sum(nums, target):
    # Метод работает для массива, отсортированного по неубыванию.
    left, right = 0, len(nums) - 1
    # Нужны два разных элемента, поэтому останавливаемся при встрече указателей.
    while left < right:
        s = nums[left] + nums[right]
        if s == target:
            return True
        if s < target:
            # Даже с самым большим доступным элементом сумма слишком мала.
            # Значит, nums[left] не подходит ни к одному из оставшихся — исключаем его.
            left += 1
        else:
            # Даже с самым маленьким доступным элементом сумма слишком велика.
            # Значит, nums[right] не подходит ни к одному из оставшихся — исключаем его.
            right -= 1
    # Все возможные пары исключены, подходящей суммы нет.
    return False

print(two_sum([2, 4, 7, 11], 15))

True


## 6. Палиндром
Сравниваю символы слева и справа, затем двигаю указатели к центру. При несовпадении возвращаю `False`. **Время O(n), память O(1)**. Семинар, стр. 58–61.

In [7]:
def is_palindrome(s):
    # Палиндром читается одинаково с обоих концов: проверяем зеркальные пары.
    left, right = 0, len(s) - 1
    while left < right:
        if s[left] != s[right]:
            return False  # Одной несовпавшей пары достаточно для отрицательного ответа.
        # Совпавшую пару больше не трогаем, сужаем проверяемую часть строки.
        left += 1
        right -= 1
    # Указатели встретились или пересеклись: все пары совпали.
    # Центральный символ, пустая строка и строка из одного символа не требуют проверки.
    return True

print(is_palindrome('шалаш'))

True


## 7. Удалить дубликаты
`fast` находит новые значения, `slow` указывает на конец уникальной части. Новые значения переписываю в тот же массив, лишний хвост удаляю через `pop()`. **Время O(n), память O(1)**.

In [8]:
def remove_duplicates(nums):
    if not nums:
        return nums

    # Массив отсортирован: одинаковые значения стоят рядом.
    # nums[0:slow + 1] — уже собранная часть без повторов; первый элемент сразу оставляем.
    slow = 0
    # fast просматривает исходные элементы, slow отмечает последнее уникальное значение.
    for fast in range(1, len(nums)):
        if nums[fast] != nums[slow]:
            # Нашли новое значение: расширяем уникальную часть и записываем его туда.
            slow += 1
            nums[slow] = nums[fast]
        # Равные значения пропускаем. Запись идёт не дальше fast, поэтому будущие данные целы.

    # Например, [1, 1, 2, 2, 3] после прохода станет [1, 2, 3, 2, 3].
    # Правильный результат уже в начале, его длина — slow + 1; лишний хвост удаляем.
    while len(nums) > slow + 1:
        nums.pop()
    return nums  # Изменён тот же массив, отдельный массив результата не создаём.

arr = [1, 1, 2, 2, 3, 3, 4]
remove_duplicates(arr)
print(arr)

[1, 2, 3, 4]


## 8. Слияние двух отсортированных списков
Выбираю меньший текущий узел и присоединяю его через `tail.next`. Когда один список закончился, присоединяю остаток второго. Новых узлов нет. **Время O(n + m), память O(1)**. Семинар, стр. 71.

In [9]:
def merge_lists(list1, list2):
    # Входные списки отсортированы; если один пуст, второй уже является ответом.
    if not list1:
        return list2
    if not list2:
        return list1

    # Отдельно выбираем голову: до этого момента у результата ещё нет последнего узла.
    if list1.val <= list2.val:
        head = list1
        list1 = list1.next  # Первый узел уже взят, указатель сдвигаем на оставшуюся часть.
    else:
        head = list2
        list2 = list2.next
    tail = head  # head сохраняет начало результата, tail будет двигаться к его концу.

    while list1 and list2:
        # Минимальный из оставшихся узлов обязательно стоит в начале одного из списков.
        if list1.val <= list2.val:
            tail.next = list1  # Присоединяем существующий узел, не создавая новый.
            list1 = list1.next
        else:
            tail.next = list2
            list2 = list2.next
        tail = tail.next  # Переводим конец результата на только что добавленный узел.

    # Один список закончился. Остаток другого уже отсортирован и не меньше tail.
    tail.next = list1 if list1 else list2
    return head

list1 = make_list([3, 6, 8])
list2 = make_list([4, 7, 9, 11])
print(to_list(merge_lists(list1, list2)))

[3, 4, 6, 7, 8, 9, 11]


## Проверка
Проверяю пустые списки, повторяющиеся значения и оба конца списка.

In [10]:
assert reverse_list(None) is None
assert to_list(reverse_list(make_list([1]))) == [1]
assert to_list(reverse_list(make_list([1, 2, 3]))) == [3, 2, 1]

assert find_middle(None) is None
assert find_middle(make_list([1, 2, 3, 4])).val == 3
assert find_middle(make_list([7])).val == 7

assert remove_value(None, 1) is None
assert to_list(remove_value(make_list([1, 1, 2, 1, 3]), 1)) == [2, 3]
assert to_list(remove_value(make_list([2, 2]), 2)) == []
assert to_list(remove_value(make_list([1, 2, 3]), 9)) == [1, 2, 3]

assert is_subsequence('', 'abc')
assert is_subsequence('abc', 'axbxc')
assert not is_subsequence('acb', 'abc')

assert not two_sum([1], 2)
assert two_sum([-3, 0, 4, 8], 5)
assert not two_sum([1, 2, 4], 10)

assert is_palindrome('')
assert is_palindrome('шалаш')
assert not is_palindrome('привет')

assert remove_duplicates([]) == []
a = [1, 1, 1]
assert remove_duplicates(a) is a and a == [1]
a = [1, 1, 2, 2, 3]
assert remove_duplicates(a) is a and a == [1, 2, 3]

assert to_list(merge_lists(None, make_list([1, 2]))) == [1, 2]
assert to_list(merge_lists(make_list([1, 2]), make_list([1, 3]))) == [1, 1, 2, 3]
assert to_list(merge_lists(make_list([3, 6, 8]), make_list([4, 7, 9, 11]))) == [3, 4, 6, 7, 8, 9, 11]
print('Все проверки пройдены')

Все проверки пройдены
